<a href="https://colab.research.google.com/github/AirtonDomiciano/automatic_model_training/blob/main/notebooks/automatic_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

**Colab:** selecione uma GPU T4 em `Ambiente de execução > Alterar tipo de ambiente de execução`. Após atualizar este notebook, use `Desconectar e excluir ambiente de execução` e execute as células em ordem. O treinamento usa um ambiente isolado Python 3.10; o Python do notebook não é substituído. A primeira instalação pode levar vários minutos. Se uma célula falhar, pare e copie o erro completo.


In [4]:
# Isolated training environment: compatible with the legacy openWakeWord pipeline.
import os
import sys
import subprocess
from pathlib import Path

def run_command(args):
    subprocess.run(args, check=True)

# uv installs Python without replacing Colab's notebook interpreter.
run_command([sys.executable, "-m", "pip", "install", "uv==0.8.22"])
run_command([sys.executable, "-m", "uv", "python", "install", "3.10"])
ENV_DIR = Path("/content/hey_pran_env")
TRAIN_PYTHON = str(ENV_DIR / "bin" / "python")
if not Path(TRAIN_PYTHON).exists():
    run_command([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(ENV_DIR)])

run_command([TRAIN_PYTHON, "-m", "ensurepip", "--upgrade"])
run_command([TRAIN_PYTHON, "-m", "pip", "install", "pip==24.0", "setuptools==69.5.1", "wheel"])
run_command([TRAIN_PYTHON, "-m", "pip", "install",
             "torch==2.1.2", "torchaudio==2.1.2", "--index-url", "https://download.pytorch.org/whl/cu118"])

# Use the release matching this notebook, rather than a moving development branch.
OWW_DIR = Path("/content/hey_pran_openwakeword_v060")
if not OWW_DIR.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", "v0.6.0",
                 "https://github.com/dscripka/openWakeWord.git", str(OWW_DIR)])
PIPER_DIR = Path("/content/piper-sample-generator")
if not PIPER_DIR.exists():
    run_command(["git", "clone", "https://github.com/rhasspy/piper-sample-generator", str(PIPER_DIR)])

requirements = [
    "numpy==1.23.5", "scipy==1.10.1", "protobuf==3.20.3",
    "tensorflow-cpu==2.8.1", "keras==2.8.0", "tensorflow-probability==0.16.0",
    "tensorflow-addons==0.16.1", "onnx==1.14.1", "onnx-tf==1.10.0",
    "onnxruntime==1.16.3", "tflite-runtime==2.14.0",
    "piper-phonemize==1.1.0", "webrtcvad==2.0.10",
    "mutagen==1.47.0", "torchinfo==1.8.0", "torchmetrics==1.2.0",
    "speechbrain==0.5.14", "audiomentations==0.33.0",
    "torch-audiomentations==0.11.0", "acoustics==0.2.6",
    "pronouncing==0.2.0", "datasets==2.14.6", "pyarrow==12.0.1",
    "huggingface-hub==0.23.5", "fsspec==2023.9.2",
    "deep-phonemizer==0.0.19", "PyYAML==6.0.1",
]
run_command([TRAIN_PYTHON, "-m", "pip", "install", *requirements])
run_command([TRAIN_PYTHON, "-m", "pip", "install", "-e", str(OWW_DIR)])
run_command([TRAIN_PYTHON, "-m", "pip", "check"])

def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size == 0:
        run_command(["wget", "--tries=3", "-O", str(destination), url])

download("https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",
         PIPER_DIR / "models/en_US-libritts_r-medium.pt")
for filename in ("embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"):
    download("https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/" + filename,
             OWW_DIR / "openwakeword/resources/models" / filename)

os.chdir("/content")
print("Training environment ready:", TRAIN_PYTHON)


Training environment ready: /content/hey_pran_env/bin/python


In [5]:
# All scientific imports and training run in the isolated Python, not Colab's Python.
import subprocess
import textwrap
import sys
from pathlib import Path

TRAIN_PYTHON = "/content/hey_pran_env/bin/python"
OWW_DIR = Path("/content/hey_pran_openwakeword_v060")
TRAIN_SCRIPT = str(OWW_DIR / "openwakeword/train.py")

def run_training_code(source):
    prelude = """import os, sys, uuid, yaml, subprocess
import numpy as np
import torch
import datasets
import scipy
from pathlib import Path
from tqdm import tqdm
"""
    try:
        # Capturamos stderr para mostrar o traceback exato do Python isolado em caso de erro
        result = subprocess.run(
            [TRAIN_PYTHON, "-u", "-c", prelude + textwrap.dedent(source)],
            capture_output=True,
            text=True,
            cwd="/content"
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr, file=sys.stderr)
        result.check_returncode()
    except subprocess.CalledProcessError as e:
        print("\n[ERRO NO AMBIENTE ISOLADO] O processo falhou. Veja os detalhes acima.")
        raise e

run_training_code("""
import tensorflow as tf
import onnx
import piper_phonemize
import openwakeword
from onnx_tf.backend import prepare
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__, "GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run again.")
""")

Python: 3.10.18 (main, Sep 18 2025, 19:50:32) [Clang 20.1.4 ]
TensorFlow: 2.8.1
PyTorch: 2.1.2+cu118 GPU available: False


[ERRO NO AMBIENTE ISOLADO] O processo falhou. Veja os detalhes acima.


Traceback (most recent call last):
  File "<string>", line 18, in <module>
RuntimeError: Select Runtime > Change runtime type > T4 GPU, then run again.



CalledProcessError: Command '['/content/hey_pran_env/bin/python', '-u', '-c', 'import os, sys, uuid, yaml, subprocess\nimport numpy as np\nimport torch\nimport datasets\nimport scipy\nfrom pathlib import Path\nfrom tqdm import tqdm\n\nimport tensorflow as tf\nimport onnx\nimport piper_phonemize\nimport openwakeword\nfrom onnx_tf.backend import prepare\nprint("Python:", sys.version)\nprint("TensorFlow:", tf.__version__)\nprint("PyTorch:", torch.__version__, "GPU available:", torch.cuda.is_available())\nif not torch.cuda.is_available():\n    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run again.")\n']' returned non-zero exit status 1.

In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
run_training_code("# Download room impulse responses collected by MIT\n# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html\n\noutput_dir = \"./mit_rirs\"\nif not os.path.exists(output_dir):\n    os.mkdir(output_dir)\nrir_dataset = datasets.load_dataset(\"davidscripka/MIT_environmental_impulse_responses\", split=\"train\", streaming=True)\n\n# Save clips to 16-bit PCM wav files\nfor row in tqdm(rir_dataset):\n    name = row['audio']['path'].split('/')[-1]\n    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))")


In [ ]:
run_training_code("## Download noise and background audio\n\n# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)\n# Download one part of the audioset .tar files, extract, and convert to 16khz\n# For full-scale training, it's recommended to download the entire dataset from\n# https://huggingface.co/datasets/agkphysics/AudioSet, and\n# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)\n\nif not os.path.exists(\"audioset\"):\n    os.mkdir(\"audioset\")\n\nfname = \"bal_train09.tar\"\nout_dir = f\"audioset/{fname}\"\nlink = \"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/\" + fname\nsubprocess.run([\"wget\", \"-O\", out_dir, link], check=True)\nsubprocess.run([\"tar\", \"-xf\", \"bal_train09.tar\"], check=True, cwd=\"audioset\")\n\noutput_dir = \"./audioset_16k\"\nif not os.path.exists(output_dir):\n    os.mkdir(output_dir)\n\n# Convert audioset files to 16khz sample rate\naudioset_dataset = datasets.Dataset.from_dict({\"audio\": [str(i) for i in Path(\"audioset/audio\").glob(\"**/*.flac\")]})\naudioset_dataset = audioset_dataset.cast_column(\"audio\", datasets.Audio(sampling_rate=16000))\nfor row in tqdm(audioset_dataset):\n    name = row['audio']['path'].split('/')[-1].replace(\".flac\", \".wav\")\n    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))\n\n# Free Music Archive dataset (https://github.com/mdeff/fma)\noutput_dir = \"./fma\"\nif not os.path.exists(output_dir):\n    os.mkdir(output_dir)\nfma_dataset = datasets.load_dataset(\"rudraml/fma\", name=\"small\", split=\"train\", streaming=True)\nfma_dataset = iter(fma_dataset.cast_column(\"audio\", datasets.Audio(sampling_rate=16000)))\n\nn_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training\nfor i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips\n    row = next(fma_dataset)\n    name = row['audio']['path'].split('/')[-1].replace(\".mp3\", \".wav\")\n    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))\n    i += 1\n    if i == n_hours*3600//30:\n        break\n")


In [ ]:
run_training_code("# Download pre-computed openWakeWord features for training and validation\n\n# training set (~2,000 hours from the ACAV100M Dataset)\n# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information\nsubprocess.run([\"wget\", \"-c\", \"https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy\"], check=True)\n\n# validation set for false positive rate estimation (~11 hours)\nsubprocess.run([\"wget\", \"-c\", \"https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy\"], check=True)")


# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey pran"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
run_training_code("""
with open("/content/hey_pran_openwakeword_v060/examples/custom_model.yml") as file:
    config = yaml.safe_load(file)
print(config)
""")


In [ ]:
run_training_code("with open(\"/content/hey_pran_openwakeword_v060/examples/custom_model.yml\") as file:\n    config = yaml.safe_load(file)\n# Modify values in the config and save a new version\n\nconfig[\"target_phrase\"] = [\"hey pran\"]\nconfig[\"model_name\"] = config[\"target_phrase\"][0].replace(\" \", \"_\")\nconfig[\"n_samples\"] = 1000\nconfig[\"n_samples_val\"] = 1000\nconfig[\"steps\"] = 10000\nconfig[\"target_accuracy\"] = 0.6\nconfig[\"target_recall\"] = 0.25\n\nconfig[\"background_paths\"] = ['./audioset_16k', './fma']  # multiple background datasets are supported\nconfig[\"false_positive_validation_data_path\"] = \"validation_set_features.npy\"\nconfig[\"feature_data_files\"] = {\"ACAV100M_sample\": \"openwakeword_features_ACAV100M_2000_hrs_16bit.npy\"}\n\nwith open('my_model.yaml', 'w') as file:\n    documents = yaml.dump(config, file)\nprint(\"Saved my_model.yaml for\", config[\"target_phrase\"])\n")


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
subprocess.run([TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--generate_clips"], check=True, cwd="/content")


In [ ]:
subprocess.run([TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--augment_clips"], check=True, cwd="/content")


In [ ]:
subprocess.run([TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--train_model"], check=True, cwd="/content")


In [ ]:
run_training_code("with open(\"my_model.yaml\") as file:\n    config = yaml.safe_load(file)\n# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly\n# If so, run this cell to retry\n\n# Manually save to tflite as this doesn't work right in colab\ndef convert_onnx_to_tflite(onnx_model_path, output_path):\n    \"\"\"Converts an ONNX version of an openwakeword model to the Tensorflow tflite format.\"\"\"\n    # imports\n    import onnx\n    import logging\n    import tempfile\n    from onnx_tf.backend import prepare\n    import tensorflow as tf\n\n    # Convert to tflite from onnx model\n    onnx_model = onnx.load(onnx_model_path)\n    tf_rep = prepare(onnx_model, device=\"CPU\")\n    with tempfile.TemporaryDirectory() as tmp_dir:\n        tf_rep.export_graph(os.path.join(tmp_dir, \"tf_model\"))\n        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, \"tf_model\"))\n        tflite_model = converter.convert()\n\n        logging.info(f\"####\\nSaving tflite mode to '{output_path}'\")\n        with open(output_path, 'wb') as f:\n            f.write(tflite_model)\n\n    return None\n\nconvert_onnx_to_tflite(f\"my_custom_model/{config['model_name']}.onnx\", f\"my_custom_model/{config['model_name']}.tflite\")\n")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!